# 02 - Creating New Optimized Versions of High-Value NASA Datasets

Below we build upon the work in check-co-hdf5-cmr.ipynb to build optimized versions of the data and a config file for testing with libraries. 

* repacked
* kerchunk original
* kerchunk repacked

To simplify this evaluation, we only use datasets which VEDA Hub has access to (which is most of the most highly used data sets).

Datasets skipped:
* if fsm_strategy and page_size were not found, likely this collection is not made up of valid HDF5 files
* oco2_L1aInSB_00018a_140703_B11000r_220105200708.h5 - compled data structure, not sure where to find data variables
* VNP03IMG.A2012019.0000.002.2020318135750.nc - don't have access to this bucket yet

In [1]:
import boto3
import fsspec
import inspect
import json
from kerchunk.hdf import SingleHdf5ToZarr
import pandas as pd
import subprocess
import yaml

# Define reprocessing functions

In [2]:
fs_read = fsspec.filesystem("s3", anon=False)

def generate_json_reference(in_filename, out_filename):
    so = dict(mode="rb", default_fill_cache=False, default_cache_type="first")
    with fs_read.open(in_filename, **so) as infile:     
        h5chunks = SingleHdf5ToZarr(in_filename)
        suffix = out_filename.split(".")[-1]
        out_filename = out_filename.replace(suffix, 'json')
        with open(out_filename, 'w') as outfile:
            outfile.write(json.dumps(h5chunks.translate()))
        return out_filename

In [3]:
processing_options = {
    "original": {
        "procssing": None,
        "input_file": None,
        "link": None
    },
    "repacked_page_4mb": {
        "processing": "h5repack -S PAGE -G 4000000",
        "input_file": "original",
        "link": None
    },   
    "kerchunk": {
        "processing": generate_json_reference,
        "input_file": "original",
        "link": None
    },
    "kerchunk_repacked_page_4mb": {
        "processing": generate_json_reference,
        "input_file": "repacked_page_4mb",
        "link": None
    }
}

# Open file with HDF5/NetCDF-4 collections

In [4]:
data_json = json.loads(open('hdf_cmr_query_results.json', 'r').read())
# remove some which had errors
filtered_list = [list(item.values())[0] for item in data_json if isinstance(item, dict)]
df = pd.DataFrame(data=filtered_list)
df = df.set_index('filename')

In [5]:
df[0:20]

,collection,version,direct_link,data_center,fsm_strategy,page_size
filename,,,,,,
ATL03_20181014000347_02350101_006_02.h5,ATL03,006,s3://nsidc-cumulus-prod-protected/ATLAS/ATL03/...,NSIDC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
ATL08_20181014001049_02350102_006_02.h5,ATL08,006,s3://nsidc-cumulus-prod-protected/ATLAS/ATL08/...,NSIDC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
AIRS.2002.08.30.225.L1B.AIRS_Rad.v5.0.0.0.G07143184358.hdf,AIRIBRAD,005,s3://gesdisc-cumulus-prod-protected/Aqua_AIRS_...,GES_DISC,Not found,Not found
OMI-Aura_ANC-OMVFPITMET_2004m1001t000305-o01132_v003-2018m1015t140716.nc4,OMVFPITMET,003,s3://gesdisc-cumulus-prod-protected/Aura_OMI_L...,GES_DISC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
OMI-Aura_ANC-OMUFPITMET_2004m1001t000305-o01132_v003-2018m1015t140708.nc4,OMUFPITMET,003,s3://gesdisc-cumulus-prod-protected/Aura_OMI_L...,GES_DISC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc,SNDRSNIML2CCPRETN,2,s3://gesdisc-cumulus-prod-protected/SNPP_Sound...,GES_DISC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
oco2_L1aInSB_00018a_140703_B11000r_220105200708.h5,OCO2_L1aIn_Sample,11r,s3://gesdisc-cumulus-prod-protected/OCO2_DATA/...,GES_DISC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
2A23.19980101.00538.7.HDF,TRMM_2A23,7,s3://gesdisc-cumulus-prod-protected/TRMM_L2/TR...,GES_DISC,Not found,Not found
3B-HHR-E.MS.MRG.3IMERG.20000601-S000000-E002959.0000.V06B.HDF5,GPM_3IMERGHHE,06,s3://gesdisc-cumulus-prod-protected/GPM_L3/GPM...,GES_DISC,H5F_FSPACE_STRATEGY_FSM_AGGR,4096


In [6]:
file_key = 'SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc'
file_data = df.loc[file_key]
short_name, version, direct_link = file_data.collection, file_data.version, file_data.direct_link
processing_options["original"]["link"] = direct_link

# Reprocess and upload

For each reprocessing option, run the reprocessing option on the input file and then upload the file and it's processing to S3.

In [7]:
destination_bucket_name = 'nasa-veda-scratch'
destination_directory = f'eodc_hdf5_experiments/{short_name}___{version}'
s3_resource = boto3.resource('s3')
s3_client = boto3.client('s3')
for processing_option, processing_data in processing_options.items():
    if processing_option == 'original':   
        # Just copy the file
        print(f"Uploading original to s3://{destination_bucket_name}/{destination_directory}/original/{file_key}")
        source_bucket = direct_link.split("/")[2]
        copy_source = {
            'Bucket': source_bucket,
            'Key': direct_link.split(f"{source_bucket}/")[-1]
        } 
        s3_client.copy(copy_source, destination_bucket_name, f"{destination_directory}/original/{file_key}")
        processing_options['original']['link'] = f"s3://{destination_bucket_name}/{destination_directory}/original/{file_key}"
    else:
        processing_cmd = processing_data['processing']
        output_file = f"{processing_option}___{file_key}"
        input_file = processing_options[processing_data["input_file"]]["link"]
        print(f"Processing {processing_cmd} on {input_file}")
        if type(processing_cmd) == str:
            cp_result = subprocess.run(["aws", "s3", "cp", direct_link, file_key], capture_output=True, text=True, check=True)
            processing_cmd_str = f"{processing_cmd} {file_key} {output_file}"
            process_result = subprocess.run(processing_cmd_str.split(" "), capture_output=True, text=True)
            # Check if there was an error
            if process_result.returncode != 0:
                print("Error:", process_result.stderr)            
        elif callable(processing_cmd) == True:
            output_file = processing_cmd(input_file, output_file)
            processing_cmd_str = inspect.getsource(processing_cmd)
            # set it to a string so we can use it for test configuration later
            processing_options[processing_option]['processing'] = processing_cmd_str
        print(f"Processing complete, uploading file to s3://{destination_bucket_name}/{destination_directory}/{output_file}")
        destination_bucket = s3_resource.Bucket(destination_bucket_name)
        destination_bucket.upload_file(output_file,  f"{destination_directory}/{output_file}")
        processing_options[processing_option]["link"] = f"s3://{destination_bucket_name}/{destination_directory}/{output_file}"
        print(f"Uploading processing cmd to s3://{destination_bucket_name}/{destination_directory}/{processing_option}_processing.txt\n")
        s3_client.put_object(
            Bucket=destination_bucket_name,
            Key=f"{destination_directory}/{processing_option}_processing.txt",
            Body=processing_cmd_str
        )

Uploading original to s3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/original/SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc
Processing h5repack -S PAGE -G 4000000 on s3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/original/SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc
Processing complete, uploading file to s3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/repacked_page_4mb___SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc
Uploading processing cmd to s3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/repacked_page_4mb_processing.txt

Processing <function generate_json_reference at 0x7f2c6e9d8c20> on s3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/original/SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc
Processing complete, uploading

In [8]:
processing_options

{'original': {'procssing': None,
  'input_file': None,
  'link': 's3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/original/SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc'},
 'repacked_page_4mb': {'processing': 'h5repack -S PAGE -G 4000000',
  'input_file': 'original',
  'link': 's3://nasa-veda-scratch/eodc_hdf5_experiments/SNDRSNIML2CCPRETN___2/repacked_page_4mb___SNDR.SNPP.CRIMSS.20120120T1254.m06.g130.L2_CLIMCAPS_RET_NSR.std.v02_28.G.200409132855.nc'},
 'kerchunk': {'processing': 'def generate_json_reference(in_filename, out_filename):\n    so = dict(mode="rb", default_fill_cache=False, default_cache_type="first")\n    with fs_read.open(in_filename, **so) as infile:     \n        h5chunks = SingleHdf5ToZarr(in_filename)\n        suffix = out_filename.split(".")[-1]\n        out_filename = out_filename.replace(suffix, \'json\')\n        with open(out_filename, \'w\') as outfile:\n            outfile.write(json.dumps(h5chunks

# Find a group and variable to test

In [9]:
import h5py
h5file = h5py.File(file_key)

In [10]:
# tunnel down through 
h5file.keys()

<KeysViewHDF5 ['air_pres', 'air_pres_h2o', 'air_pres_h2o_nsurf', 'air_pres_lay', 'air_pres_lay_bnds', 'air_pres_lay_nsurf', 'air_pres_nsurf', 'air_temp', 'air_temp_dof', 'air_temp_err', 'air_temp_qc', 'asc_flag', 'asc_node_local_solar_time', 'asc_node_lon', 'asc_node_tai93', 'atrack', 'attitude', 'attitude_lbl', 'aux', 'ave_kern', 'bnds_1d', 'ch4_dof', 'ch4_mmr_midtrop', 'ch4_mmr_midtrop_err', 'ch4_mmr_midtrop_qc', 'cld_frac', 'cld_frac_err', 'cld_frac_qc', 'cld_lay', 'cld_lay_lbl', 'cld_top_pres', 'cld_top_pres_err', 'cld_top_pres_qc', 'cld_top_temp', 'cld_top_temp_err', 'cld_top_temp_qc', 'co2_dof', 'co_dof', 'co_mmr_midtrop', 'co_mmr_midtrop_err', 'co_mmr_midtrop_qc', 'fov', 'fov_land_frac', 'fov_lat', 'fov_lat_bnds', 'fov_lon', 'fov_lon_bnds', 'fov_obs_id', 'fov_poly', 'fov_surf_alt', 'fov_surf_alt_sdev', 'gp_hgt', 'gp_hgt_err', 'gp_hgt_qc', 'h2o_liq_mol_lay', 'h2o_liq_mol_lay_err', 'h2o_liq_mol_lay_qc', 'h2o_liq_tot', 'h2o_liq_tot_err', 'h2o_liq_tot_qc', 'h2o_vap_dof', 'h2o_vap_to

In [11]:
group = '/'
variable = 'surf_temp'

# Create tests object

In [12]:
test_dict = {
    "collection": short_name,
    "group": group,
    "variable": variable,
    "files": processing_options
}

In [13]:
# Write the dictionary to a file
with open (f'../../h5cloud/file_configs/{short_name}_{version}.yaml', 'w') as file:
    yaml.dump(test_dict, file, default_flow_style=False)


In [14]:
!ls ../../h5cloud/file_configs/

ATL03_006.yaml	       GSSTF_NCEP_3.yaml	     OMVFPITMET_003.yaml
ATL08_006.yaml	       LPRM_AMSR2_D_SOILM3_001.yaml  SNDRSNIML2CCPRETN_2.yaml
GPM_3IMERGHHE_06.yaml  OMUFPITMET_003.yaml	     atl03.yml
